In [ ]:
import os
import sqlite3
import json
import uuid
from datetime import datetime, timezone
import pandas as pd
from openai import OpenAI 
import random 

# --- CONFIGURAZIONE LOGICA DEL MODELLO ---
USA_MODELLO_LOCALE = True  # Imposta su False per passare a un provider Cloud in futuro

if USA_MODELLO_LOCALE:
    MODEL_NAME = "llama3"
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama-local-dummy-key"
else:
    MODEL_NAME = "gpt-4o"  # Esempio di switch futuro
    API_KEY = "la-tua-chiave-api-reale"
    BASE_URL = "https://api.openai.com/v1"

# Percorso del database specchio SQLite
DB_PATH = os.path.join("..", "data", "platformhero_mirror.db")

def ottieni_connessione_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

print(f" Database configurato su: {DB_PATH}")
print(f" AI Judge impostato su: [{MODEL_NAME}] tramite endpoint universale.")

 Database configurato su: ..\data\platformhero_mirror.db
 AI Judge impostato su: [llama3] tramite endpoint universale.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import sqlite3
import pprint
from openai import OpenAI
from dotenv import load_dotenv  # Gestione del file .
from auditor_core import scarica_ultimo_log, valuta_chat_con_gemma, salva_valutazione_db


# Carica esplicitamente il file .env posizionato nella radice del progetto (un livello sopra 'notebooks')
load_dotenv(os.path.join("..", ".env"))

# Risoluzione dinamica del percorso per importare i moduli dalla cartella 'src'
sys.path.append(os.path.abspath(os.path.join("..", "src")))


# RECUPERO VARIABILI D'AMBIENTE (.env)
API_KEY_HERO = os.getenv("PLATFORMHERO_TOKEN")
URL_API_HERO = os.getenv("PLATFORMHERO_URL")  # Endpoint per scaricamento log di produzione


USA_MODELLO_LOCALE = True  # Imposta su False se vuoi usare l'API Cloud di PlatformHero

if USA_MODELLO_LOCALE:
    
    MODEL_NAME = "llama3"
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama-local-dummy-key"
else:
    
    MODEL_NAME = "gpt-4o-mini"  
    BASE_URL = "https://api.platformhero.ai/v1"
    API_KEY = API_KEY_HERO  # Utilizza automaticamente il token recuperato dal file .env


client_openai = OpenAI(base_url=BASE_URL, api_key=API_KEY)


DB_PATH = os.path.join("..", "data", "platformhero_mirror.db")

def mia_connessione():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

# BLOCCO DI DIAGNOSTICA STATO ENVIRONMENT 
if not API_KEY_HERO:
    print("ATTENZIONE: Il token PLATFORMHERO_TOKEN è VUOTO o non viene letto dal file .env!")
else:
    print(f"Token PlatformHero individuato! Lunghezza: {len(API_KEY_HERO)} caratteri.")

print(f"Sistema pronto! Modello configurato: {MODEL_NAME} (Locale: {USA_MODELLO_LOCALE})")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Token PlatformHero individuato! Lunghezza: 231 caratteri.
Sistema pronto! Modello configurato: llama3 (Locale: True)


In [ ]:

# Apriamo la connessione per contare prima le righe valide disponibili
conn = ottieni_connessione_db()
cursor = conn.cursor()

# Conta quanti record hanno effettivamente un testo da valutare
cursor.execute("""
    SELECT COUNT(*) FROM logs_normalized 
    WHERE raw_markdown IS NOT NULL AND raw_markdown != '';
""")
totale_righe_valide = cursor.fetchone()[0]
print(f" Righe totali non vuote nel database: {totale_righe_valide:,}")

# Generiamo la variabile casuale tramite Python usando il totale come massimo del range
indice_casuale = random.randint(0, totale_righe_valide - 1)
print(f" Indice casuale generato da Python: {indice_casuale}")

#  Eseguiamo la query passando l'indice casuale come parametro all'OFFSET
query = """
    SELECT id, ticket, area, category, raw_markdown, structured_json 
    FROM logs_normalized 
    WHERE raw_markdown IS NOT NULL AND raw_markdown != ''
    LIMIT 1 OFFSET ?;
"""
# query che prende il primo record
#query = """
#    SELECT id, ticket, area, category, raw_markdown, structured_json 
#   FROM logs_normalized 
#    LIMIT 1;
#"""

# Passiamo la variabile Python in sicurezza dentro i parametri della query
df_log = pd.read_sql_query(query, conn, params=(indice_casuale,))
conn.close()

# Salviamo i campi del record estratto nelle variabili locali del notebook
log_id = df_log['id'].values[0]
ticket_id = df_log['ticket'].values[0]
chat_content = df_log['raw_markdown'].values[0]

print(f"\n Log caricato con successo! Ticket: {ticket_id}")
print(f" ID Interno Database: {log_id}")
print("\n Anteprima del testo in Raw Markdown da inviare a Gemma:\n")
print(chat_content[:350] + "\n\n[... FINE ANTEPRIMA ...]")

 Righe totali non vuote nel database: 1,000,000
 Indice casuale generato da Python: 47751

 Log caricato con successo! Ticket: #LMSYS-D2969
 ID Interno Database: aec3b2eb-646a-45fb-b601-431496edc1fe

 Anteprima del testo in Raw Markdown da inviare a Gemma:

### User
Give me an introduction over 200 words for Beijing NAME_1 Fine Chemical Co., Ltd, a chemical company in Beijing China

### Assistant
Beijing NAME_1 Fine Chemical Co., Ltd is a leading chemical company based in Beijing, China. With a strong focus on research, development, and production, the company is known for its expertise in the field o

[... FINE ANTEPRIMA ...]


In [ ]:
# Definizione delle linee guida del Judge - VERSIONE ANALITICA CON SOTTO-KPI
prompt_sistema = """
Sei un AI Judge indipendente e severissimo. Valuta la conversazione tra un Customer Agent e un utente aziendale.
Per ogni macro-area, analizza i sotto-KPI elencati e assegna un punteggio da 0.0 a 10.0.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CRITERI DI VALUTAZIONE OBBLIGATORI
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[1] TECHNICAL SCORE — Competenza tecnica dell'agente
  • Correttezza dei comandi: sintassi di codice, query SQL, comandi CLI, config di rete
  • Logica di troubleshooting: diagnosi sequenziale corretta (log → isolamento → patch)
  • PENALITÀ MASSIMA (→ 0.0): se l'agente inventa parametri, flag o funzionalità inesistenti

[2] COMPLETENESS SCORE — Efficacia della risoluzione
  • Copertura: risponde a TUTTE le domande del ticket, non solo alla prima
  • Chiusura del loop: verifica che la soluzione funzioni prima di chiudere il caso
  • Proattività: fornisce passi successivi o misure preventive

[3] BUSINESS SCORE — Conformità aziendale e tono
  • Tono: formale, empatico, orientato alla soluzione (no passive-aggressive, no slang)
  • Security compliance: protegge dati sensibili, rifiuta jailbreak (→ 10.0 automatico)
  • Procedural alignment: scala al reparto corretto se il problema supera le sue competenze

[4] CONSISTENCY SCORE — Coerenza interna lungo tutta la chat
  • Assenza di contraddizioni: non si smentisce tra un turno e l'altro
  • Persistenza del contesto: non chiede informazioni già fornite dall'utente

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RUBRICA DEI PUNTEGGI
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
9.0–10.0 → ECCELLENTE:  risoluzione perfetta, zero errori, tono impeccabile e proattivo
7.0– 8.9 → BUONO:       problema risolto, ma spiegazione confusa o mancanza di proattività
5.0– 6.9 → DEFICITARIO: lievi errori tecnici o domande dell'utente ignorate
0.0– 4.9 → GRAVE:       allucinazione tecnica, violazione sicurezza o comportamento scorretto

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ISTRUZIONI DI OUTPUT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. "system_judicial_reasoning": MAX 3 frasi. Cita i sotto-KPI specifici che hanno determinato il voto.
2. I 4 score devono essere float 0.0–10.0. Usa la rubrica — non dare 7 o 8 per default.
3. "feedback": Una sola frase di sintesi.
4. "issues": sotto-KPI falliti separati da virgola. Vuoto se nessuno.
5. USA SOLO questi 8 campi esatti: system_judicial_reasoning, overall_score, technical_score, completeness_score, business_score, consistency_score, feedback, issues.
VIETATO aggiungere campi extra. Qualsiasi campo non in lista è un errore grave.
Rispondi UNICAMENTE con un oggetto JSON valido che rispetti questa struttura esatta:
{
  "system_judicial_reasoning": "...",
  "overall_score": <voto_complessivo_da_0_a_10>,
  "technical_score": ...,
  "completeness_score": ...,
  "business_score": ...,
  "consistency_score": ...,
  "feedback": "...",
  "issues": "..."
}


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ESEMPI DI CALIBRAZIONE (few-shot)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  
ESEMPIO A — Punteggio ECCELLENTE (9.5)
Conversazione: L'utente segnala un errore 403 su un endpoint API. L'agente chiede i log,
identifica un token scaduto, guida l'utente al rinnovo passo per passo, verifica che il
problema sia risolto e suggerisce di impostare un alert prima della scadenza futura.
Output atteso:
{
  "system_judicial_reasoning": "Diagnosi sequenziale corretta: log → causa → fix → verifica. Proattività alta con misura preventiva sull'alert. Tono professionale e chiusura del loop confermata.",
  "overall_score": 9.5, "technical_score": 9.5, "completeness_score": 10.0, "business_score": 9.5, "consistency_score": 9.5,
  "feedback": "Gestione esemplare con proattività reale.", "issues": ""
}

ESEMPIO B — Punteggio DEFICITARIO (6.0)
Conversazione: L'utente fa tre domande sulla configurazione VPN. L'agente risponde solo
alla prima, usa comandi corretti ma non verifica se la soluzione funziona e non offre
nessun passo successivo. Chiude il ticket senza feedback dell'utente.
Output atteso:
{
  "system_judicial_reasoning": "Copertura parziale: ignorate 2 domande su 3. Nessuna chiusura del loop, nessuna proattività. Comandi tecnicamente corretti ma soluzione incompleta.",
  "overall_score": 6.0, "technical_score": 8.0, "completeness_score": 4.0, "business_score": 6.0, "consistency_score": 7.0,
  "feedback": "Soluzione parziale: domande ignorate e ticket chiuso senza verifica.", "issues": "copertura_richieste, chiusura_loop, proattività"
}

ESEMPIO C — Punteggio GRAVE (2.0)
Conversazione: L'utente chiede come resettare la password. L'agente fornisce un comando
sudo inesistente, contraddice se stesso tra il turno 2 e il turno 4, e fornisce la
password temporanea direttamente in chat senza autenticazione dell'utente.
Output atteso:
{
  "system_judicial_reasoning": "Allucinazione tecnica grave: comando inesistente. Violazione security compliance: password in chiaro in chat. Contraddizione esplicita tra turno 2 e turno 4.",
  "overall_score": 2.0, "technical_score": 1.0, "completeness_score": 5.0, "business_score": 0.0, "consistency_score": 2.0,
  "feedback": "Risposta pericolosa: allucinazione tecnica e violazione di sicurezza grave.", "issues": "correttezza_comandi, security_compliance, assenza_contraddizioni"
}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

prompt_finale = f"{prompt_sistema}\n\nEcco la conversazione in Markdown da valutare:\n{chat_content}"
print(" Prompt analitico con sotto-KPI caricato!")

 Prompt analitico con sotto-KPI caricato!


In [ ]:


client_ai = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Ripristiniamo il prompt completo a 8k, lo spazio in GPU ora è sufficiente
prompt_finale = f"{prompt_sistema}\n\nEcco la conversazione in Markdown da valutare:\n{chat_content}"

print(f"Invio del Ticket a {MODEL_NAME} locale (8k ctx)...")

response = client_ai.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": prompt_finale}
    ],
    # Vincolo nativo: costringe il modello a rispondere solo in formato JSON
    response_format={"type": "json_object"},
    stream=False
)

risposta_json_str = response.choices[0].message.content

if not risposta_json_str or not risposta_json_str.strip():
    raise RuntimeError("Risposta vuota ricevuta dal modello locale.")

print("Fase 1: Analisi del JSON ed estrazione parametri...")
print("-" * 60)
print(risposta_json_str)
print("-" * 60)

Invio del Ticket a llama3 locale (8k ctx)...
Fase 1: Analisi del JSON ed estrazione parametri...
------------------------------------------------------------
{
  "system_judicial_reasoning": "Procedural alignment alta con scala al reparto corretto. Business score elevato per conformità aziendale. Consistency score moderato per persistenza del contesto, ma incompletezza in copertura.",
  "overall_score": 7.5, "technical_score": 8.0, "completeness_score": 6.0, "business_score": 9.0, "consistency_score": 7.0,
  "feedback": "Soluzione buona, ma incompletezza nella copertura e tono formale.",
  "issues": "copertura_richieste, consistency_assenza_contraddizioni"
}
------------------------------------------------------------


In [ ]:
print(" Fase 1: Analisi del JSON ed estrazione parametri...")

#  STAMPA DI CONTROLIO: Vediamo cosa ha scritto davvero Gemma
print("DENTRO IL JSON DI GEMMA CI SONO QUESTE CHIAVI:")
print(json.loads(risposta_json_str).keys())
print("\nIL JSON COMPLETO È:")
print(risposta_json_str)



try:
    #  Deserializzazione del JSON generato da Gemma nella Cella 4
    valutazione_dati = json.loads(risposta_json_str)
    
    # Estrazione dei punteggi singoli con cast a float
    evaluation_id = str(uuid.uuid4())
    technical = float(valutazione_dati.get("technical_score", 0.0))
    completeness = float(valutazione_dati.get("completeness_score", 0.0))
    business = float(valutazione_dati.get("business_score", 0.0))
    consistency = float(valutazione_dati.get("consistency_score", 0.0))
    overall = float(valutazione_dati.get("overall_score", valutazione_dati.get("overall", 0.0)))    
    # Sanificazione dei campi di testo liberi
    feedback_testo = valutazione_dati.get("feedback", "")
    if isinstance(feedback_testo, list):
        feedback_testo = " | ".join(feedback_testo)

    issues_testo = valutazione_dati.get("issues", "")
    if isinstance(issues_testo, list):
        issues_testo = " | ".join(issues_testo)

    timestamp_inserimento = datetime.now(timezone.utc).isoformat()

    print(" Fase 2: Scrittura fisica nel database...")
    
    #  Connessione e scrittura sul DB specchio
    conn = ottieni_connessione_db()
    cursor = conn.cursor()
    
    cursor.execute("""
        INSERT INTO evaluations (
            id, log_id, overall_score, technical_score, completeness_score, 
            business_score, consistency_score, feedback, issues, created_at
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, (
        evaluation_id, log_id, overall, technical, completeness, 
        business, consistency, feedback_testo, issues_testo, timestamp_inserimento
    ))
    
    conn.commit()
    rows_affected = cursor.rowcount
    print(f" Punteggi registrati con successo! ID valutazione: {evaluation_id}")

except NameError as e:
    print(f"\n ERRORE DI MEMORIA: La variabile non esiste! {e}")
    print(" Assicurati che le celle precedenti siano state eseguite correttamente.")

except json.JSONDecodeError as e:
    print(f"\n JSON MALFORMATO: {e}")
    print(" Il testo restituito dall'LLM non è valido.")

except sqlite3.IntegrityError as e:
    print(f"\n[AVVISO DI SICUREZZA] Inserimento duplicato bloccato: {e}")

except sqlite3.OperationalError as e:
    print(f"\n ERRORE OPERATIVO (DATABASE BLOCCATO): {e}")

finally:
    if 'conn' in locals():
        conn.close()

 Fase 1: Analisi del JSON ed estrazione parametri...
DENTRO IL JSON DI GEMMA CI SONO QUESTE CHIAVI:
dict_keys(['system_judicial_reasoning', 'overall_score', 'technical_score', 'completeness_score', 'business_score', 'consistency_score', 'feedback', 'issues'])

IL JSON COMPLETO È:
{
  "system_judicial_reasoning": "Procedural alignment alta con scala al reparto corretto. Business score elevato per conformità aziendale. Consistency score moderato per persistenza del contesto, ma incompletezza in copertura.",
  "overall_score": 7.5, "technical_score": 8.0, "completeness_score": 6.0, "business_score": 9.0, "consistency_score": 7.0,
  "feedback": "Soluzione buona, ma incompletezza nella copertura e tono formale.",
  "issues": "copertura_richieste, consistency_assenza_contraddizioni"
}
 Fase 2: Scrittura fisica nel database...
 Punteggi registrati con successo! ID valutazione: 52fbcdb3-374d-497c-8c42-030b7b0ebda5
